# Week 3: Named Entity Extraction

This notebook extracts structured entities from real estate listing remarks, including bedrooms, bathrooms, price, square footage, and amenities.

In [77]:
import pandas as pd
import sys

sys.path.append("..")

from scripts.entity_extractor import EntityExtractor

## Load Cleaned Listing Data

The cleaned listing dataset from Week 2 is used for entity extraction.

In [78]:
df = pd.read_csv("../data/processed/listing_sample_cleaned.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1000, 8)


,L_ListingID,L_Address,L_City,beds,baths,price,remarks,cleaned_remarks
0,1189451542,36612 Desert Tortoise Road,Murrieta,3.0,3.0,635000,"Welcome to 36612 Desert Tortoise Road, a beaut...","Welcome to 36612 Desert Tortoise Road, a beaut..."
1,1159634800,904 Amistad Court C,El Cajon,2.0,2.0,409000,PRICE REDUCED! on this IMMACULATE turnkey cond...,PRICE REDUCED! on this IMMACULATE turnkey cond...
2,1144651476,1255 Delresto Drive,Beverly Hills,4.0,4.0,14495000,"Majestic Gated Italian Villa, An Architectural...","Majestic Gated Italian Villa, An Architectural..."
3,1179077960,16855 Redwood Lodge Road,Los Gatos,4.0,3.0,1195000,Tucked into the trees of the Santa Cruz Mounta...,Tucked into the trees of the Santa Cruz Mounta...
4,1190215696,5252 Agnes 2,Valley Village,2.0,2.0,599000,A magnificently remodeled corner unis with 3 (...,A magnificently remodeled corner unis with 3 (...


## Initialize Entity Extractor

The `EntityExtractor` uses regex patterns for numeric entities and the Week 1 taxonomy for amenity detection.

In [79]:
extractor = EntityExtractor("../data/processed/taxonomy.json")

## Extract Entities
#### entity: a meaningful piece of information you want to pull out of text.

Extract bedrooms, bathrooms, price, square footage, and amenities from each listing remark.

In [80]:
df["entities"] = df["cleaned_remarks"].apply(extractor.extract_all)

df[
    ["L_ListingID", "cleaned_remarks", "entities"]
].head(10)

,L_ListingID,cleaned_remarks,entities
0,1189451542,"Welcome to 36612 Desert Tortoise Road, a beaut...","{'bedrooms': 3.0, 'bathrooms': 2.5, 'price': N..."
1,1159634800,PRICE REDUCED! on this IMMACULATE turnkey cond...,"{'bedrooms': None, 'bathrooms': None, 'price':..."
2,1144651476,"Majestic Gated Italian Villa, An Architectural...","{'bedrooms': None, 'bathrooms': None, 'price':..."
3,1179077960,Tucked into the trees of the Santa Cruz Mounta...,"{'bedrooms': 4.0, 'bathrooms': 2.5, 'price': N..."
4,1190215696,A magnificently remodeled corner unis with 3 (...,"{'bedrooms': None, 'bathrooms': None, 'price':..."
5,1158643901,New Vinyl fencing is installed!! Affordable Du...,"{'bedrooms': None, 'bathrooms': None, 'price':..."
6,1183471208,"Welcome to the residences of El Dorado Manor, ...","{'bedrooms': 2.0, 'bathrooms': None, 'price': ..."
7,1176097870,Welcome to this beautiful two-story home nestl...,"{'bedrooms': 4.0, 'bathrooms': 3.0, 'price': N..."
8,1175988469,**Price reduction*** GREAT VALUE !!! Bright en...,"{'bedrooms': 2.0, 'bathrooms': 2.0, 'price': N..."
9,1145948172,SPACIOUS RANCH RETREAT WITH ENDLESS POTENTIAL ...,"{'bedrooms': None, 'bathrooms': None, 'price':..."


## Create Labeled Dataset Sample

A sample of 250 listing remarks is selected for entity-span labeling and evaluation.

In [81]:
labeled_df = df[
    ["L_ListingID", "cleaned_remarks"]
].sample(
    n=250,
    random_state=42
).copy()

labeled_df = labeled_df.reset_index(drop=True)

print("Labeled dataset size:", len(labeled_df))
labeled_df.head()

Labeled dataset size: 250


,L_ListingID,cleaned_remarks
0,1169367198,Welcome to this charming single-story home nes...
1,1183143372,Beautifully remodeled and completely move-in r...
2,1178547165,Sophisticated Contemporary Living on the Wilsh...
3,1177750953,LOOK NO FURTHER... This stunning two-story poo...
4,1156892963,"Welcome to this beautifully maintained, move-i..."


In [82]:
labeled_df.to_csv(
    "../data/processed/week3_labeled_sample.csv",
    index=False
)

print("Saved 250 remarks to:")
print("../data/processed/week3_labeled_sample.csv")

Saved 250 remarks to:
../data/processed/week3_labeled_sample.csv


## Add Entity Spans

Entity spans record the exact text location (start and end) of extracted entities within each listing remark.

In [83]:
import re

def extract_entity_spans(text):
    spans = []

    patterns = {
        "BEDROOMS": r"\d+(?:\.\d+)?\s*[- ]?\s*(?:bedrooms?|beds?|br|bd)\b",
        "BATHROOMS": r"\d+(?:\.\d+)?\s*[- ]?\s*(?:bathrooms?|baths?|ba)\b",
        "PRICE": r"\$[\d,]{5,}",
        "SQFT": r"\d[\d,]*\s*(?:square feet|sq\.?\s*ft\.?|sqft)\b"
    }

    for label, pattern in patterns.items():
        for match in re.finditer(pattern, text, re.IGNORECASE):
            spans.append({
                "text": match.group(),
                "label": label,
                "start": match.start(),
                "end": match.end()
            })

    for amenity in extractor.amenities:
        pattern = rf"\b{re.escape(amenity)}\b"

        for match in re.finditer(pattern, text, re.IGNORECASE):
            spans.append({
                "text": match.group(),
                "label": "AMENITY",
                "start": match.start(),
                "end": match.end()
            })

    return sorted(spans, key=lambda x: x["start"])

In [84]:
labeled_df["entity_spans"] = labeled_df["cleaned_remarks"].apply(
    extract_entity_spans
)

labeled_df[
    ["L_ListingID", "cleaned_remarks", "entity_spans"]
].head()

,L_ListingID,cleaned_remarks,entity_spans
0,1169367198,Welcome to this charming single-story home nes...,"[{'text': 'single-story', 'label': 'AMENITY', ..."
1,1183143372,Beautifully remodeled and completely move-in r...,"[{'text': 'guest parking', 'label': 'AMENITY',..."
2,1178547165,Sophisticated Contemporary Living on the Wilsh...,"[{'text': '2-bedroom', 'label': 'BEDROOMS', 's..."
3,1177750953,LOOK NO FURTHER... This stunning two-story poo...,"[{'text': 'two-story', 'label': 'AMENITY', 'st..."
4,1156892963,"Welcome to this beautifully maintained, move-i...","[{'text': 'gated community', 'label': 'AMENITY..."


Span example: {
    "text": "3-bedroom",
    "label": "BEDROOMS",
    "start": 76,
    "end": 85
}

In [86]:
labeled_df = labeled_df.rename(
    columns={"entity_spans": "candidate_spans"}
)

labeled_df.to_csv(
    "../data/processed/week3_labeled_sample.csv",
    index=False
)

print("Saved 250 remarks with candidate entity spans.")

Saved 250 remarks with candidate entity spans.


## Create Evaluation Test Set

A small subset of the labeled remarks is selected for manual correction and evaluation of the entity extractor.

In [87]:
test_df = labeled_df.sample(
    n=50,
    random_state=42
).copy()

test_df = test_df.reset_index(drop=True)

test_df["true_spans"] = test_df["candidate_spans"]

print("Evaluation set size:", len(test_df))
test_df.head()

Evaluation set size: 50


,L_ListingID,cleaned_remarks,candidate_spans,true_spans
0,1156994571,"Welcome to easy, resort-style living in this b...","[{'text': '2-bedroom', 'label': 'BEDROOMS', 's...","[{'text': '2-bedroom', 'label': 'BEDROOMS', 's..."
1,1189742033,**MAJOR PRICE IMPROVEMENT! Exceptional Value S...,"[{'text': 'single-story', 'label': 'AMENITY', ...","[{'text': 'single-story', 'label': 'AMENITY', ..."
2,1184646768,"Sitting beautifully behind an automatic gate, ...","[{'text': '3-bedroom', 'label': 'BEDROOMS', 's...","[{'text': '3-bedroom', 'label': 'BEDROOMS', 's..."
3,1177163841,Welcome to 315 Seascape Resort Drive in the pr...,"[{'text': '1-bedroom', 'label': 'BEDROOMS', 's...","[{'text': '1-bedroom', 'label': 'BEDROOMS', 's..."
4,1183398560,Located within the highly sought-after Wisebur...,"[{'text': '3-bedroom', 'label': 'BEDROOMS', 's...","[{'text': '3-bedroom', 'label': 'BEDROOMS', 's..."


In [88]:
print("Evaluation set size:", len(test_df))

Evaluation set size: 50


## Generate Predictions

The entity extractor is applied to the evaluation set so its predictions can be compared with the labeled entity spans.

In [89]:
test_df["predicted_entities"] = test_df["cleaned_remarks"].apply(
    extractor.extract_all
)

test_df[
    ["L_ListingID", "cleaned_remarks", "predicted_entities"]
].head()

,L_ListingID,cleaned_remarks,predicted_entities
0,1156994571,"Welcome to easy, resort-style living in this b...","{'bedrooms': 2.0, 'bathrooms': 2.0, 'price': N..."
1,1189742033,**MAJOR PRICE IMPROVEMENT! Exceptional Value S...,"{'bedrooms': 5.0, 'bathrooms': 4.0, 'price': 1..."
2,1184646768,"Sitting beautifully behind an automatic gate, ...","{'bedrooms': 3.0, 'bathrooms': 2.0, 'price': N..."
3,1177163841,Welcome to 315 Seascape Resort Drive in the pr...,"{'bedrooms': 1.0, 'bathrooms': 1.0, 'price': N..."
4,1183398560,Located within the highly sought-after Wisebur...,"{'bedrooms': 3.0, 'bathrooms': 2.0, 'price': N..."


## Ground-Truth Entity Spans

The automatically generated candidate spans are used as a starting point. The evaluation subset is manually checked and corrected to create ground-truth entity spans.

In [90]:
import copy

test_df["true_spans"] = test_df["candidate_spans"].apply(copy.deepcopy)

test_df[
    ["L_ListingID", "cleaned_remarks", "true_spans"]
].head()

,L_ListingID,cleaned_remarks,true_spans
0,1156994571,"Welcome to easy, resort-style living in this b...","[{'text': '2-bedroom', 'label': 'BEDROOMS', 's..."
1,1189742033,**MAJOR PRICE IMPROVEMENT! Exceptional Value S...,"[{'text': 'single-story', 'label': 'AMENITY', ..."
2,1184646768,"Sitting beautifully behind an automatic gate, ...","[{'text': '3-bedroom', 'label': 'BEDROOMS', 's..."
3,1177163841,Welcome to 315 Seascape Resort Drive in the pr...,"[{'text': '1-bedroom', 'label': 'BEDROOMS', 's..."
4,1183398560,Located within the highly sought-after Wisebur...,"[{'text': '3-bedroom', 'label': 'BEDROOMS', 's..."


In [91]:
for i in range(10):
    print(f"\nROW {i}")
    print("Listing ID:", test_df.loc[i, "L_ListingID"])
    print("\nTEXT:")
    print(test_df.loc[i, "cleaned_remarks"])
    print("\nSPANS:")
    
    for span in test_df.loc[i, "true_spans"]:
        print(span)

    print("\n" + "=" * 100)


ROW 0
Listing ID: 1156994571

TEXT:
Welcome to easy, resort-style living in this beautifully maintained 2-bedroom, 2-bathroom home with an enclosed den/office/flex space, perfect for a home office, hobby room, or guest retreat. Nestled within a desirable gated 55+ golf course community, this home offers both comfort and convenience in a well-connected setting. Enjoy a bright and functional layout complemented by a spacious, shaded backyard ideal for relaxing or entertaining. The property also features generous side yards and mature fruit trees, creating a private outdoor oasis. Ideally located across from a tranquil pond, the home offers added privacy with no front neighbors. Take advantage of the incredible walkability - just a short stroll to the golf course entrance and the Monarch Restaurant. Community amenities elevate the lifestyle with access to two swimming pools, a spa, library, and a recreation facility. Stay engaged with monthly community magazines that highlight events, cl

In [ ]:
# ROW 0: Remove generic phrase "Community amenities"
test_df.at[0, "true_spans"] = [
    s for s in test_df.at[0, "true_spans"]
    if s["text"] != "Community amenities"
]

# ROW 1: Remove lot square footage and overlapping duplicate amenity matches
remove_row1 = {
    "15325 square feet",
    "pool",      # already captured as "sparkling pool"
    "patio"      # already captured as "covered patio"
}

test_df.at[1, "true_spans"] = [
    s for s in test_df.at[1, "true_spans"]
    if s["text"] not in remove_row1
]

# ROW 5: "lot" refers to a parking lot, not a property amenity
test_df.at[5, "true_spans"] = [
    s for s in test_df.at[5, "true_spans"]
    if s["text"] != "lot"
]

# ROW 7: Remove overlapping shorter matches and Add missed bathroom entity
spans = test_df.at[7, "true_spans"]
spans = [
    s for s in spans
    if not (
        (s["text"] == "lot" and s["start"] == 314)
        or (s["text"] == "garage" and s["start"] == 679)
    )
]
spans.append({
    "text": "3-full-bathroom",
    "label": "BATHROOMS",
    "start": 126,
    "end": 141
})

test_df.at[7, "true_spans"] = sorted(
    spans,
    key=lambda x: x["start"]
)

# ROW 9: Add bedroom/bathroom entities written as words
spans = test_df.at[9, "true_spans"]
spans.append({
    "text": "two spacious bedrooms",
    "label": "BEDROOMS",
    "start": 173,
    "end": 194
})

spans.append({
    "text": "two bathrooms",
    "label": "BATHROOMS",
    "start": 199,
    "end": 212
})

test_df.at[9, "true_spans"] = sorted(
    spans,
    key=lambda x: x["start"]
)

print("Rows 0-9 corrected.")

Rows 0-9 corrected.


In [93]:
for i in range(10, 20):
    print(f"\nROW {i}")
    print("Listing ID:", test_df.loc[i, "L_ListingID"])
    print("\nTEXT:")
    print(test_df.loc[i, "cleaned_remarks"])
    print("\nSPANS:")

    for span in test_df.loc[i, "true_spans"]:
        print(span)

    print("\n" + "=" * 100)


ROW 10
Listing ID: 1177733487

TEXT:
Welcome to this beautifully remodeled turnkey home nestled in the highly admired community of Sierra Madre. Perfect as a starter home or for anyone seeking comfort, style, and convenience, this inviting residence offers the ideal blend of modern updates and small-town charm. Featuring newly remodeled bedrooms and updated bathrooms, this home boasts a beautifully remodeled kitchen that's truly a chef's dream, complete with modern finishes and kitchen appliances. Step outside to a private backyard, perfect for summer barbecues, relaxing evenings, or hosting family and friends. The warm, cozy atmosphere makes this home an ideal place to create lasting memories. A versatile bonus room provides endless possibilities - it can easily be used as a den, a home office, a playroom, or a creative studio to fit your lifestyle. Conveniently located just minutes from local grocery stores, the post office, charming restaurants, neighborhood bars, and the beloved M

In [ ]:
# ROW 11 : Remove lot square footage
test_df.at[11, "true_spans"] = [
    s for s in test_df.at[11, "true_spans"]
    if s["text"] != "10523 square feet"
]


# ROW 14: Remove lot square footage
test_df.at[14, "true_spans"] = [
    s for s in test_df.at[14, "true_spans"]
    if s["text"] != "4,900 square feet"
]


# ROW 15: Add missed price, bedroom, and bathroom entities
spans = test_df.at[15, "true_spans"]
spans.extend([
    {
        "text": "5,299,000",
        "label": "PRICE",
        "start": 15,
        "end": 24
    },
    {
        "text": "6 ensuite bedrooms",
        "label": "BEDROOMS",
        "start": 493,
        "end": 511
    },
    {
        "text": "8 total bathrooms",
        "label": "BATHROOMS",
        "start": 513,
        "end": 530
    },
    {
        "text": "5,299,000",
        "label": "PRICE",
        "start": 1312,
        "end": 1321
    }
])

test_df.at[15, "true_spans"] = sorted(
    spans,
    key=lambda x: x["start"]
)

# ROW 16: Remove overlapping shorter "yard"
test_df.at[16, "true_spans"] = [
    s for s in test_df.at[16, "true_spans"]
    if not (s["text"] == "yard" and s["start"] == 490)
]


# ROW 17: Remove overlapping shorter "patio"
test_df.at[17, "true_spans"] = [
    s for s in test_df.at[17, "true_spans"]
    if not (s["text"] == "patio" and s["start"] == 863)
]


# ROW 18
spans = test_df.at[18, "true_spans"]

spans.extend([
    {
        "text": "3,200 square foot",
        "label": "SQFT",
        "start": 108,
        "end": 125
    },
    {
        "text": "three bedrooms",
        "label": "BEDROOMS",
        "start": 803,
        "end": 817
    },
    {
        "text": "fourth bedroom",
        "label": "BEDROOMS",
        "start": 843,
        "end": 857
    }
])

test_df.at[18, "true_spans"] = sorted(
    spans,
    key=lambda x: x["start"]
)


# ROW 19
spans = test_df.at[19, "true_spans"]

# Remove overlapping shorter "yard"
spans = [
    s for s in spans
    if not (s["text"] == "yard" and s["start"] == 716)
]

# Add missed bathroom count
spans.append({
    "text": "3 full bathrooms",
    "label": "BATHROOMS",
    "start": 79,
    "end": 95
})

test_df.at[19, "true_spans"] = sorted(
    spans,
    key=lambda x: x["start"]
)

print("Rows 10-19 corrected.")

Rows 10-19 corrected.


In [95]:
for i in range(20, 30):
    print(f"\nROW {i}")
    print("Listing ID:", test_df.loc[i, "L_ListingID"])

    print("\nTEXT:")
    print(test_df.loc[i, "cleaned_remarks"])

    print("\nSPANS:")
    for span in test_df.loc[i, "true_spans"]:
        print(span)

    print("\n" + "=" * 100)


ROW 20
Listing ID: 1159279309

TEXT:
Welcome to 1405 Zion Way, a beautifully remodeled mountain retreat nestled in the peaceful community of Pine Mountain Club. This stunning 3-bedroom, 2-bathroom home offers the perfect blend of modern upgrades and serene natural surroundings. Completely remodeled, the home features a fresh, updated interior designed for both comfort and style. The open and inviting layout is filled with natural light, creating a warm and airy atmosphere throughout. The upgraded kitchen and living spaces flow seamlessly, making it ideal for entertaining or enjoying quiet evenings surrounded by nature. Step outside and take in the breathtaking forest setting, where towering pines and crisp mountain air provide a true escape from the hustle of city life. Whether you're sipping coffee on the deck or unwinding after a long day, this home delivers a peaceful, private retreat year-round. Located in the highly desirable Pine Mountain Club community, residents enjoy access t

In [ ]:
# Helper for adding a missed span safely
def add_span(row_idx, phrase, label):
    text = test_df.at[row_idx, "cleaned_remarks"]
    start = text.find(phrase)

    if start != -1:
        span = {
            "text": phrase,
            "label": label,
            "start": start,
            "end": start + len(phrase)
        }

        if span not in test_df.at[row_idx, "true_spans"]:
            test_df.at[row_idx, "true_spans"].append(span)

        test_df.at[row_idx, "true_spans"] = sorted(
            test_df.at[row_idx, "true_spans"],
            key=lambda x: x["start"]
        )


# ROW 21
# "room for a pool" means a pool does not currently exist
test_df.at[21, "true_spans"] = [
    s for s in test_df.at[21, "true_spans"]
    if not (s["text"] == "pool" and s["start"] == 1555)
]

# Written-out square footage and bathroom count
add_span(21, "four hundred square foot", "SQFT")
add_span(21, "two more baths", "BATHROOMS")


# ROW 22
# "spa-like bathroom" is descriptive, not an actual spa
test_df.at[22, "true_spans"] = [
    s for s in test_df.at[22, "true_spans"]
    if s["text"].lower() != "spa"
]


# ROW 23
# Remove overlapping shorter solar match
test_df.at[23, "true_spans"] = [
    s for s in test_df.at[23, "true_spans"]
    if not (s["text"].lower() == "solar" and s["start"] == 995)
]


# ROW 25
# 5000 sqft describes the community amenity center, not the home
test_df.at[25, "true_spans"] = [
    s for s in test_df.at[25, "true_spans"]
    if s["text"] != "5000 square feet"
]

add_span(25, "three secondary bedrooms", "BEDROOMS")


# ROW 26
# Remove lot square footage
test_df.at[26, "true_spans"] = [
    s for s in test_df.at[26, "true_spans"]
    if s["text"] != "6903 square feet"
]


# ROW 27
# Remove overlapping shorter patio match
test_df.at[27, "true_spans"] = [
    s for s in test_df.at[27, "true_spans"]
    if not (s["text"] == "patio" and s["start"] == 685)
]


# ROW 28
spans = test_df.at[28, "true_spans"]

# Remove descriptive "spa-inspired"
# Remove shorter patio because "covered patio" already captures it
spans = [
    s for s in spans
    if not (
        (s["text"].lower() == "spa" and s["start"] == 1113)
        or (s["text"] == "patio" and s["start"] == 1839)
    )
]

test_df.at[28, "true_spans"] = spans

add_span(28, "three bedrooms", "BEDROOMS")
add_span(28, "three-and-a-half bathrooms", "BATHROOMS")
add_span(28, "two additional bedrooms", "BEDROOMS")


# ROW 29
# Remove overlapping shorter garage match
test_df.at[29, "true_spans"] = [
    s for s in test_df.at[29, "true_spans"]
    if not (s["text"] == "garage" and s["start"] == 495)
]


print("Rows 21-29 corrected.")

Rows 21-29 corrected.


In [97]:
for i in range(30, 40):
    print(f"\nROW {i}")
    print("Listing ID:", test_df.loc[i, "L_ListingID"])

    print("\nTEXT:")
    print(test_df.loc[i, "cleaned_remarks"])

    print("\nSPANS:")
    for span in test_df.loc[i, "true_spans"]:
        print(span)

    print("\n" + "=" * 100)


ROW 30
Listing ID: 1177084844

TEXT:
Welcome to 1250 Rolling Hill Ct, a spacious single-story home tucked away on a quiet cul-de-sac. Situated on a generous corner lot, this 4-bedroom, 2-bathroom home offers comfortable living space with a functional layout designed for everyday living. Large windows fill the home with natural light, while multiple living areas provide flexibility for gathering, relaxing or entertaining. The spacious game room features a built-in bar with a wine fridge and opens directly to the backyard, creating the ultimate space for hosting friends and family. Step outside to a sparkling pool and expansive deck, perfect for summer barbecues, outdoor dining, or simply unwinding at the end of the day. The generous backyard offers room to garden, play, or personalize, making this home an incredible opportunity for buyers looking to create their dream outdoor retreat. Conveniently located near parks, shopping, dining, schools, and commuter routes, this home combines a 

In [ ]:
# ROW 30
# Remove overlapping shorter matches
test_df.at[30, "true_spans"] = [
    s for s in test_df.at[30, "true_spans"]
    if not (
        (s["text"] == "lot" and s["start"] == 127)
        or (s["text"] == "pool" and s["start"] == 578)
    )
]


# ROW 32
# Remove generic phrase "community amenities"
test_df.at[32, "true_spans"] = [
    s for s in test_df.at[32, "true_spans"]
    if s["text"].lower() != "community amenities"
]


# ROW 33
# "spa-inspired soaking tub" is descriptive, not an actual spa
test_df.at[33, "true_spans"] = [
    s for s in test_df.at[33, "true_spans"]
    if not (s["text"].lower() == "spa" and s["start"] == 1284)
]


# ROW 34
spans = test_df.at[34, "true_spans"]

# Remove overlapping shorter garage/patio matches
spans = [
    s for s in spans
    if not (
        (s["text"] == "garage" and s["start"] == 432)
        or (s["text"] == "patio" and s["start"] == 730)
    )
]

test_df.at[34, "true_spans"] = spans

# Add missed bathroom count
add_span(34, "three full bathrooms", "BATHROOMS")


# ROW 35
spans = test_df.at[35, "true_spans"]

# Remove false amenity: "Garden" is part of Garden Grove
# Remove $10,000 because it is a design-center credit, not listing price
spans = [
    s for s in spans
    if not (
        s["text"] == "Garden"
        or s["text"] == "$10,000"
    )
]

test_df.at[35, "true_spans"] = spans

# Add written-out bedroom/bathroom entities
add_span(35, "three bedroom", "BEDROOMS")
add_span(35, "half bathroom", "BATHROOMS")


# ROW 36
# Remove overlapping "lot" because "corner lot" already captures it
test_df.at[36, "true_spans"] = [
    s for s in test_df.at[36, "true_spans"]
    if not (s["text"] == "lot" and s["start"] == 68)
]


# ROW 37
# Remove generic phrase and overlapping shorter patio
test_df.at[37, "true_spans"] = [
    s for s in test_df.at[37, "true_spans"]
    if not (
        s["text"].lower() == "community amenities"
        or (s["text"] == "patio" and s["start"] == 971)
    )
]


# ROW 39
spans = test_df.at[39, "true_spans"]

# Remove partial price, lot sqft, and overlapping shorter amenities
spans = [
    s for s in spans
    if not (
        (s["label"] == "PRICE" and s["start"] == 16)
        or s["text"] == "14705 square feet"
        or (s["text"] == "floor plan" and s["start"] == 281)
        or (s["text"] == "yard" and s["start"] == 692)
    )
]

test_df.at[39, "true_spans"] = spans

# Add the full price text
add_span(39, "$199,000.00", "PRICE")


print("Rows 30-39 corrected.")

Rows 30-39 corrected.


In [99]:
for i in range(40, 50):
    print(f"\nROW {i}")
    print("Listing ID:", test_df.loc[i, "L_ListingID"])

    print("\nTEXT:")
    print(test_df.loc[i, "cleaned_remarks"])

    print("\nSPANS:")
    for span in test_df.loc[i, "true_spans"]:
        print(span)

    print("\n" + "=" * 100)


ROW 40
Listing ID: 1177959219

TEXT:
Beautifully updated 2-bedroom, 2-bathroom residence offering 1193 square feet. of refined living space in the highly sought-after Skyline Condominiums. This north-facing home features an expansive private terrace with unobstructed Downtown city views, creating the perfect setting for relaxing or entertaining. The open-concept floor plan showcases wood flooring, updated kitchen with granite countertops, stainless steel appliances, and a breakfast bar that flows seamlessly into the spacious living and dining areas. Generous closet space, elegant marble bathrooms, and an in-unit washer and dryer provide both comfort and convenience. The residence also includes two assigned tandem parking spacesa rare find in Downtown LA. Residents enjoy 24-hour security and concierge service, a resort-style pool and spa, fitness center, racquetball court, saunas, clubhouse, outdoor BBQ areas, and beautifully landscaped grounds. Ideally located in the heart of South Pa

In [ ]:
# ROW 41
# Add the later explicit bedroom/bathroom counts
add_span(41, "four bedrooms", "BEDROOMS")
add_span(41, "three full bathrooms", "BATHROOMS")


# ROW 42
# Remove incorrect partial bathroom match
test_df.at[42, "true_spans"] = [
    s for s in test_df.at[42, "true_spans"]
    if s["text"] != "4 bathroom"
]

# Add the actual bathroom descriptions
add_span(42, "3 (1 full and 2-3/4) baths", "BATHROOMS")
add_span(42, "2 (3/4 bathroom) bathroom", "BATHROOMS")


# ROW 43
# "Recreation Area" refers to Hansen Dam Recreation Area, not a feature of the property
test_df.at[43, "true_spans"] = [
    s for s in test_df.at[43, "true_spans"]
    if s["text"] != "Recreation Area"
]


# ROW 44
# These bedrooms/bathrooms are only part of FUTURE ADU plans, so don't treat them as existing property entities
test_df.at[44, "true_spans"] = [
    s for s in test_df.at[44, "true_spans"]
    if not (
        s["text"] == "2-bedroom"
        or s["text"] == "1-bathroom"
    )
]

# ROW 49
# "spa inspired" describes the bathroom style, not an actual spa amenity
test_df.at[49, "true_spans"] = [
    s for s in test_df.at[49, "true_spans"]
    if not (s["text"].lower() == "spa" and s["start"] == 1131)
]

print("Rows 40-49 corrected.")
print("All 50 evaluation rows reviewed.")

Rows 40-49 corrected.
All 50 evaluation rows reviewed.


In [102]:
test_df.to_csv(
    "../data/processed/week3_evaluation_set.csv",
    index=False
)

print("Saved evaluation set:", len(test_df), "rows")

Saved evaluation set: 50 rows


## Evaluation

True Positives (model predicted an entity and it really was an entity): 348

False Positives (model predicted something that should not have been labeled): 41

False Negatives (something should have been detected, but the extractor missed it): 25

Precision: 0.8946 (89.46% predicted entities were correct)

Recall:    0.9330 (model found roughly 93.3% true entities)

F1 score:        0.9134

## Error Analysis

F1 score = 0.9134

Most mistakes happened because the extractor follows fixed regex/taxonomy rules instead of truly understanding context.

Top failure patterns:
- Lot square footage is mistaken for home living-area sqft
- Community/building sqft is mistaken for individual home sqft
- Descriptive phrases like “spa-inspired” are sometimes mistaken for an actual spa amenity.
- Phrases like “room for a pool” are sometimes mistaken for an existing pool.
- Overlapping amenities such as “sparkling pool” + “pool” or “covered patio” + “patio” can be counted as two entities instead of one.
- Written-out numbers such as “three bedrooms” may be missed because the regex mainly looks for digits.
- Credits or incentives can be mistaken for listing prices, such as “Seller is offering $10,000 toward the design center.”